# Extraction de données exactes de pdfs

## Importation des bibliothèques

In [1]:
from IPython.display import JSON

import json

from unstructured_client import UnstructuredClient
from unstructured_client.models import shared
from unstructured_client.models.errors import SDKError

from unstructured.partition.html import partition_html
from unstructured.partition.pdf import partition_pdf
from unstructured.staging.base import dict_to_elements, elements_to_json

In [2]:
import nltk
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

print(nltk.data.path)

['/Users/estebantarin/nltk_data', '/Library/Frameworks/Python.framework/Versions/3.11/nltk_data', '/Library/Frameworks/Python.framework/Versions/3.11/share/nltk_data', '/Library/Frameworks/Python.framework/Versions/3.11/lib/nltk_data', '/usr/share/nltk_data', '/usr/local/share/nltk_data', '/usr/lib/nltk_data', '/usr/local/lib/nltk_data']


[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/estebantarin/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/estebantarin/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


## Récupération du fichier pdf et de ses tableaux

In [3]:
from unstructured.partition.pdf import partition_pdf

#filename = "KID PRIIPS - Feeder Boscalt - Class C (français).pdf"
filename = "KID PRIIPS - Feeder Boscalt - Class C (français).pdf"


elements = partition_pdf(filename)

elements

On transforme chaque élément extrait du pdf de base (paragraphe, tableau, image, ...) en une liste de ditionnaires, que l'on va ensuite passer au format json. 

In [4]:
elements_dict = [e.to_dict() for e in elements]

output = json.dumps(elements_dict, indent=2)

print(output)

[
  {
    "type": "Title",
    "element_id": "aca66c548281a17fb82b9a424e996281",
    "text": "Document d'Informations Cl\u00e9s (KID)",
    "metadata": {
      "coordinates": {
        "points": [
          [
            31.92,
            57.799999999999955
          ],
          [
            31.92,
            73.39999999999998
          ],
          [
            264.36,
            73.39999999999998
          ],
          [
            264.36,
            57.799999999999955
          ]
        ],
        "system": "PixelSpace",
        "layout_width": 595.2,
        "layout_height": 841.68
      },
      "filename": "KID PRIIPS - Feeder Boscalt - Class C (franc\u0327ais).pdf",
      "languages": [
        "eng"
      ],
      "last_modified": "2024-10-14T15:02:05",
      "page_number": 1,
      "filetype": "application/pdf"
    }
  },
  {
    "type": "Title",
    "element_id": "6c4c4322eb328253b7d944711b7f51f8",
    "text": "OBJET",
    "metadata": {
      "coordinates": {
       

In [5]:
unique_types = set()

for item in elements_dict : 
    unique_types.add(item["type"])

print(unique_types)

{'UncategorizedText', 'Footer', 'Title', 'ListItem', 'NarrativeText'}


La cellule suivante permet d'extraire uniquement les tableaux reconnus par unstructured comme PDF. Noys faisons en sorte que le code essaie de plus de reconnaitre par lui même la structure du tableau pour qu'il reconnaisse les liens entre chaque case. 

In [6]:
elements = partition_pdf(filename=filename,
                         infer_table_structure=True,
                         strategy='hi_res',
           )

tables = [el for el in elements if el.category == "Table"]

print(tables[0].text)
print(tables[0].metadata.text_as_html)

PRIIP Manufacturer : Compagnie Benjamin de Rothschild Management (Luxembourg) S.A. Adresse : 11-13 Rue Jean Fischbach L-3372 Leudelange Grand-Duché de Luxembourg
<table><thead><tr><th>PRIIP</th><th>Manufacturer :</th><th>Compagnie Benjamin de Rothschild Management (Luxembourg) S.A. ed</th></tr></thead><tbody><tr><td rowspan="3">Adresse :</td><td></td><td>11-13 Rue Jean Fischbach</td></tr><tr><td></td><td>L-3372 Leudelange</td></tr><tr><td></td><td>Grand-Duché de Luxembourg</td></tr></tbody></table>


## Passage en html des metadonnees du tableau

On récupère le fichier en html pour que l'on puisse conserve la structure du fichier explicitement. 

In [7]:
table_html = tables[0].metadata.text_as_html

In [8]:
from io import StringIO 
from lxml import etree

parser = etree.XMLParser(remove_blank_text=True)
file_obj = StringIO(table_html)
tree = etree.parse(file_obj, parser)
print(etree.tostring(tree, pretty_print=True).decode())

<table>
  <thead>
    <tr>
      <th>PRIIP</th>
      <th>Manufacturer :</th>
      <th>Compagnie Benjamin de Rothschild Management (Luxembourg) S.A. ed</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td rowspan="3">Adresse :</td>
      <td/>
      <td>11-13 Rue Jean Fischbach</td>
    </tr>
    <tr>
      <td/>
      <td>L-3372 Leudelange</td>
    </tr>
    <tr>
      <td/>
      <td>Grand-Duch&#233; de Luxembourg</td>
    </tr>
  </tbody>
</table>



In [9]:
from IPython.core.display import HTML
HTML(table_html)

## Utilisation d'Ollama (llama3.2)

In [10]:
from langchain_community.chat_models import ChatOllama
from langchain_core.documents import Document
from langchain.chains.summarize import load_summarize_chain

In [11]:
ChatOllama??

Init signature:
ChatOllama(
    *args: Any,
    name: Optional[str] = None,
    cache: Union[langchain_core.caches.BaseCache, bool, NoneType] = None,
    verbose: bool = <factory>,
    callbacks: Union[list[langchain_core.callbacks.base.BaseCallbackHandler], langchain_core.callbacks.base.BaseCallbackManager, NoneType] = None,
    tags: Optional[list[str]] = None,
    metadata: Optional[dict[str, Any]] = None,
    custom_get_token_ids: Optional[Callable[[str], list[int]]] = None,
    base_url: str = 'http://localhost:11434',
    model: str = 'llama2',
    mirostat: Optional[int] = None,
    mirostat_eta: Optional[float] = None,
    mirostat_tau: Optional[float] = None,
    num_ctx: Optional[int] = None,
    num_gpu: Optional[int] = None,
    num_thread: Optional[int] = None,
    num_predict: Optional[int] = None,
    repeat_last_n: Optional[int] = None,
    repeat_penalty: Optional[float] = None,
    temperature: Optional[float] = None,
    stop: Optional[List[str]] = None,
    tfs_z: O

In [13]:
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

# Prompt personnalisé pour poser une question précise
question = "Quel est le PRIIP Manufacturer ?" 

prompt = PromptTemplate(
    input_variables=["table_html", "question"],
    template="Voici un document HTML : {table_html}. Répondez à la question suivante : {question}"
)

llm = ChatOllama(model="llama3.2")
llm_chain = LLMChain(prompt=prompt, llm=llm)

output = llm_chain.run(table_html=table_html, question=question)

In [14]:
print(output)

Bonjour !

D'après le document HTML fourni, on peut voir que :

* Le PRIIP est "Compagnie Benjamin de Rothschild Management (Luxembourg) S.A. ed"
* La manufacturer est également indiquée sous la forme "Compagnie Benjamin de Rothschild Management (Luxembourg) S.A. ed"

Il semblerait donc qu'il y ait une doublon dans le document HTML, où les informations sur le PRIIP et la manufacturer sont identiques.

Pour obtenir des informations plus précises, il faudrait examiner le contexte du document et déterminer ce que signifie "ed" en fin de phrase. Cela pourrait être un acronyme ou une abréviation qui nécessite des informations supplémentaires pour être complètement compris.

Si vous avez plus d'informations sur le document ou si vous souhaitez obtenir plus de détails sur le contexte, n'hésitez pas à me les partager !


## Utilisation de LM Studios

In [16]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:1234/v1", api_key="not-needed")

completion = client.chat.completions.create(
    model="local-model",
    messages=[
        {
            "role": "user",
            "content": f"Voici le fichier HTML : {table_html}. Maintenant, {question}"
        },
    ]
)

# Afficher uniquement le contenu du message généré
print(completion.choices[0].message.content)


Le PRIIP (Premium Rate Indicator, Plan Revisé Immediate et Permanent) est une séquence de caractères qui identifie le fabricant ou l'éditeur d'une activité téléphonique ou de services en ligne. Dans votre exemple, le PRIIP est :

`Compagnie Benjamin de Rothschild Management (Luxembourg) S.A. ed`

C'est-à-dire que le fabricant ou l'éditeur de cette activité est :

`Compagnie Benjamin de Rothschild Management (Luxembourg) S.A.`

L'ajout "ed" en fin de séquence est une indication d'édition ou de modification du contenu.
